# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_after.png" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env
from IPython.display import display, Markdown

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

OPENAI_API_KEY=<not set>
LANGSMITH_API_KEY=****954e
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=****ials


In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [3]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [6]:
import os
from langchain_aws import ChatBedrock
from langchain.agents import create_agent

llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    model_kwargs=dict(temperature=0),
    region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1")
)

agent = create_agent(
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [7]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================

[{'type': 'text', 'text': "<thinking> To determine the total on Frank Harris's last invoice, I first need to identify the most recent invoice for him. I will start by querying the invoices to find the most recent one for Frank Harris. </thinking>\n"}, {'type': 'tool_use', 'name': 'execute_sql', 'input': {'query': 'SELECT id, customer_name, invoice_date FROM invoices ORDER BY invoice_date DESC LIMIT 5;'}, 'id': 'tooluse_6_os7IswQzGMiJafUOUB_g'}]
Tool Calls:
  execute_sql (tooluse_6_os7IswQzGMiJafUOUB_g)
 Call ID: tooluse_6_os7IswQzGMiJafUOUB_g
  Args:
    query: SELECT id, customer_name, invoice_date FROM invoices ORDER BY invoice_date DESC LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.Op

In [8]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

[{'type': 'text', 'text': '<thinking>Based on the given request, it\'s unclear which table or data we are referring to when we talk about "titles". To proceed, I need to find the table with potential title data and then display the first few rows of that table to confirm the column name that contains the titles.</thinking>\n'}, {'type': 'tool_use', 'name': 'execute_sql', 'input': {'query': "SELECT name FROM sqlite_master WHERE type='table';"}, 'id': 'tooluse_FwNxHZI0QAGfredm7zWDzw'}]
Tool Calls:
  execute_sql (tooluse_FwNxHZI0QAGfredm7zWDzw)
 Call ID: tooluse_FwNxHZI0QAGfredm7zWDzw
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table';
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',),

## Add memory

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

In [10]:
import os
from langchain_aws import ChatBedrock
from langchain.agents import create_agent

llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    model_kwargs=dict(temperature=0),
    region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1")
)

agent = create_agent(
    model=llm,
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
    
)

In [11]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================

[{'type': 'text', 'text': "<thinking> To find the total amount on Frank Harris's last invoice, I need to:\n\n1. Identify the most recent invoice for Frank Harris.\n2. Extract the total amount from that invoice.\nI'll need to execute a query to find the most recent invoice for Frank Harris and then extract the total amount from it. </thinking>\n"}, {'type': 'tool_use', 'name': 'execute_sql', 'input': {'query': "SELECT id, customer_name, invoice_date, total_amount FROM Invoices WHERE customer_name = 'Frank Harris' ORDER BY invoice_date DESC LIMIT 1"}, 'id': 'tooluse_GxQBJ22tT6GkASpGWwQ5kQ'}]
Tool Calls:
  execute_sql (tooluse_GxQBJ22tT6GkASpGWwQ5kQ)
 Call ID: tooluse_GxQBJ22tT6GkASpGWwQ5kQ
  Args:
    query: SELECT id, customer_name, invoice_date, total_amount FROM I

In [12]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

[{'type': 'text', 'text': '<thinking> It seems the user has asked for the titles from the last invoice of Frank Harris. I will need to join the "Invoice" table with another table that contains invoice line items to get the titles of the items. Based on the schema information, it\'s likely that the "InvoiceLine" table contains the information about each item purchased on an invoice, and it can be joined with the "Track" table to get the titles of the items. I will need to query the "InvoiceLine" and "Track" tables to get the titles of the items on Frank Harris\'s last invoice. </thinking> '}, {'type': 'tool_use', 'name': 'execute_sql', 'input': {'query': 'SELECT InvoiceLine.InvoiceId, Track.Name FROM InvoiceLine JOIN Track ON InvoiceLine.TrackId = Track.TrackId WHERE InvoiceLine.InvoiceId = 374'}, 'id': 

## Try your own queries
Now that there is memory, check the agents recall!

In [13]:
question = "Your Question Here?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

Your Question Here?
================================== Ai Message ==================================

It looks like you didn't provide a specific question. If you have a topic or area of interest you'd like to explore, feel free to ask! Here are a few examples of what you might inquire about:

1. **History:** What was the impact of the Industrial Revolution on society?
2. **Science:** How do vaccines work to protect against viruses?
3. **Technology:** What are the implications of artificial intelligence on future jobs?
4. **Environment:** How can we mitigate the effects of climate change?
5. **Literature:** What are some common themes in Shakespeare's plays?

Please provide a specific question, and I'll do my best to give you a detailed and informative answer!
